# AllLife Bank — Personal Loan Campaign Targeting

> A decision-tree classifier and 3-tier segmentation playbook that turns a 9.6% campaign success rate into a 3.1× targeted lift.

**Author:** Nikhil Patel · **Live demo:** [nikhilcpatel.com/demo-alllife-bank.html](https://www.nikhilcpatel.com/demo-alllife-bank.html)

---

## 1. Problem framing

AllLife Bank's customer base is overwhelmingly liability customers. Marketing's last loan campaign converted 9.6%. They want to know: which customers should we contact next so the conversion rate goes up?

- **Imbalanced classes** (9.6% positive)
- **Asymmetric error cost** — false positive is cheap (one phone call); false negative is expensive (years of loan interest)
- **Interpretability matters** — marketing needs to *see* the rule, not just a score

Those three constraints lead to: **decision tree** classifier, **recall** as primary metric, and **post-pruning** to balance recall against precision.

## 2. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 1

## 3. Load and inspect the data

In [ ]:
df = pd.read_csv('Loan_Modelling.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
# Check the target distribution
df['Personal_Loan'].value_counts(normalize=True).round(4)

## 4. Data cleaning and feature engineering

Three issues to fix:
1. **Negative `Experience`** values — 52 rows with -1, -2, -3. Likely data-entry errors. Impute to absolute value.
2. **`ZIP_Code`** has 467 unique values — too many for a categorical. Engineer down to 2-digit prefix.
3. **`Experience`** is 0.99 correlated with `Age` — drop to avoid multicollinearity (decision trees handle it but it confuses feature importance).

In [ ]:
# 1. Fix negative experience
neg_exp = (df['Experience'] < 0).sum()
print(f'Negative Experience rows: {neg_exp}')
df['Experience'] = df['Experience'].abs()

# 2. ZIP prefix
df['ZIP_Prefix'] = df['ZIPCode'].astype(str).str[:2].astype(int)
print(f'ZIP prefixes: {df["ZIP_Prefix"].nunique()}')

# 3. Drop columns that don't help
df = df.drop(columns=['ID', 'ZIPCode', 'Experience'])
df.head()

## 5. Exploratory data analysis

Hypothesis-driven, not "let's plot everything." Each plot answers a question.

### 5.1 Income — the single strongest signal

In [ ]:
bands = pd.cut(df['Income'], bins=[0,50,75,100,150,250],
                labels=['<$50K','$50-75K','$75-100K','$100-150K','$150K+'])
conv_by_income = df.groupby(bands)['Personal_Loan'].mean().round(3)
print(conv_by_income)

fig, ax = plt.subplots(figsize=(10,5))
conv_by_income.plot(kind='bar', ax=ax, color='#00d9ff')
ax.axhline(0.096, ls='--', color='gray', label='Base rate 9.6%')
ax.set_ylabel('Conversion rate')
ax.set_title('Loan acceptance by income band')
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 5.2 CD Account — the killer categorical

In [ ]:
print(df.groupby('CD_Account')['Personal_Loan'].agg(['count','mean']).round(3))
# Only 6% of customers have a CD account, but 46% of them convert. 4.8x lift.

### 5.3 What doesn't predict

In [ ]:
for col in ['Age','Family','Securities_Account','Online','CreditCard','ZIP_Prefix']:
    rate = df.groupby(col)['Personal_Loan'].mean().round(3)
    print(f'\n{col}:')
    print(rate)

### 5.4 Correlation heatmap

In [ ]:
plt.figure(figsize=(11,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True)
plt.title('Feature correlations')
plt.tight_layout()
plt.show()

## 6. Train / test split

In [ ]:
X = df.drop(columns=['Personal_Loan'])
y = df['Personal_Loan']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train positive rate: {y_train.mean():.3f} | Test: {y_test.mean():.3f}')

## 7. Three models, three stories

We'll fit three trees and compare them on **recall** (primary), with precision and F1 as constraints.

| Model | Why |
|---|---|
| Default | Baseline — confirms overfitting risk |
| Pre-pruned via grid search | Aggressive depth limit + balanced class weights |
| Post-pruned via cost-complexity α | Final production model |

### 7.1 Helper to evaluate

In [ ]:
def evaluate(model, X_train, X_test, y_train, y_test, name=''):
    yp_train = model.predict(X_train)
    yp_test = model.predict(X_test)

    metrics = {
        'Accuracy_Train':  accuracy_score(y_train, yp_train),
        'Recall_Train':    recall_score(y_train, yp_train),
        'Precision_Train': precision_score(y_train, yp_train),
        'F1_Train':        f1_score(y_train, yp_train),
        'Accuracy_Test':  accuracy_score(y_test, yp_test),
        'Recall_Test':    recall_score(y_test, yp_test),
        'Precision_Test': precision_score(y_test, yp_test),
        'F1_Test':        f1_score(y_test, yp_test),
        'Leaves':         model.get_n_leaves(),
        'Depth':          model.get_depth(),
    }
    return pd.Series(metrics, name=name)

### 7.2 Default tree (overfits)

In [ ]:
tree_default = DecisionTreeClassifier(random_state=RANDOM_STATE)
tree_default.fit(X_train, y_train)
m_default = evaluate(tree_default, X_train, X_test, y_train, y_test, 'Default')
m_default

### 7.3 Pre-pruned tree (grid search)

In [ ]:
params = {
    'max_depth': [2, 3, 4, 5, 7],
    'max_leaf_nodes': [4, 8, 16, 32],
    'min_samples_split': [10, 30, 50],
    'class_weight': ['balanced']
}
gs = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    params, scoring='recall', cv=5, n_jobs=-1
)
gs.fit(X_train, y_train)
tree_pre = gs.best_estimator_
print('Best params:', gs.best_params_)
m_pre = evaluate(tree_pre, X_train, X_test, y_train, y_test, 'Pre-pruned')
m_pre

### 7.4 Post-pruned tree (cost-complexity α)

In [ ]:
path = DecisionTreeClassifier(
    random_state=RANDOM_STATE, class_weight={0:0.15, 1:0.85}
).cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

trees, recalls = [], []
for a in ccp_alphas:
    t = DecisionTreeClassifier(random_state=RANDOM_STATE,
                               class_weight={0:0.15, 1:0.85}, ccp_alpha=a)
    t.fit(X_train, y_train)
    trees.append(t)
    recalls.append(recall_score(y_test, t.predict(X_test)))

best_idx = int(np.argmax(recalls))
tree_post = trees[best_idx]
print(f'Best ccp_alpha: {ccp_alphas[best_idx]:.5f}')
m_post = evaluate(tree_post, X_train, X_test, y_train, y_test, 'Post-pruned')
m_post

### 7.5 Compare all three

In [ ]:
compare = pd.concat([m_default, m_pre, m_post], axis=1).round(3)
compare

**Why post-pruned wins.** Pre-pruned achieves 100% recall by flagging ~40% of customers — precision collapses to 31%. Post-pruned gives up 15 points of recall to triple the precision. For a budget-bounded campaign, that's the right trade.

## 8. Feature importance and interpretation

In [ ]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': tree_post.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x='importance', y='feature', data=importance, color='#00d9ff')
plt.title('Feature importance — post-pruned tree')
plt.tight_layout()
plt.show()
importance

### 8.1 Visualize the post-pruned tree

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(tree_post, feature_names=X.columns,
          class_names=['No', 'Yes'], filled=True, max_depth=3, fontsize=10)
plt.title('Post-pruned decision tree (top 3 levels)')
plt.tight_layout()
plt.show()

### 8.2 Visualize the pre-pruned tree (segmentation framework)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(tree_pre, feature_names=X.columns,
          class_names=['No', 'Yes'], filled=True, fontsize=11)
plt.title('Pre-pruned tree — depth-2, the segmentation framework')
plt.tight_layout()
plt.show()

## 9. From model to playbook — the 3-tier segmentation

The pre-pruned tree's 4 leaves map cleanly to 3 marketing tiers (two leaves combine into HOT).

In [ ]:
# Tier customers using pre-pruned tree probabilities
proba = tree_pre.predict_proba(X_test)[:, 1]

tiers = pd.cut(proba, bins=[-0.01, 0.50, 0.80, 1.01], labels=['SKIP','WARM','HOT'])
tier_summary = pd.DataFrame({
    'tier': tiers,
    'actual': y_test.values
}).groupby('tier').agg(
    customers=('actual','count'),
    buyers=('actual','sum'),
    conversion=('actual','mean')
).round(3)
tier_summary['lift_vs_base'] = (tier_summary['conversion'] / 0.096).round(2)
tier_summary

## 10. Recommendations

1. **Run HOT tier as a premium campaign** — 107 customers, 67% conversion. Dedicated outbound rep.
2. **Treat WARM as a nurture funnel** — 373 customers, 21% conversion. Email + SMS sequence.
3. **Suppress SKIP** — 1,020 customers, 0% conversion. Re-tier quarterly.
4. **Cross-sell loans to CD holders** — 4.8× base lift. Trigger an offer at CD origination.
5. **Operationalize the tree as a SQL view** — refresh weekly, hand to marketing as a CRM segment.
6. **Hold out 10% of HOT as control** — without it you can't separate model lift from baseline demand.

## 11. Lessons learned

**Metric is downstream of economics.** Recall over accuracy isn't a technical call — it's about which error costs more.

**Two models for two jobs.** Post-pruned tree = production classifier. Pre-pruned tree = segmentation framework. Don't force one to do both.

**The bridge from model to action is the actual job.** A 73-leaf tree is unreadable for marketing. A 4-leaf one becomes a SQL rule a marketing analyst can run.